# Deploying Agents with Gradio

*Notebook 30*

Your agent only runs where your notebook runs. Gradio wraps it in a real chat interface you can share, from a local demo to a hosted Space.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

MODEL = "gpt-5-mini"

print("✅ Ready!")

try:
    import gradio as gr
    print("✅ Gradio ready")
except ImportError:
    print("❌ Gradio not found. Run: pip install gradio==6.20.0")

---

## 🎯 The Problem

Your agent works in a notebook. But notebooks aren't for users.

Gradio gives you a real chat interface with one function, deployable to Hugging Face Spaces.

---

## 🖥️ Part 1: What Gradio Is

Gradio is a Python library for building web UIs around machine learning models and AI agents.

It's a standard choice for AI demos because:

- A working chat interface is a small Python wrapper: one chat function plus `gr.ChatInterface`

- `gr.ChatInterface` handles the conversation loop, message history, and UI automatically

- Gradio apps deploy directly to Hugging Face Spaces: free hosting tiers, a shareable URL, no servers to manage (free Spaces sleep when idle)

---

## 💬 Part 2: Wrap an Agent in `gr.ChatInterface`

These next four parts build the same app step by step: start with Part 2, then layer on streaming, status updates, and error handling.

Part 6's deployment template combines them all.

The minimal Gradio app: define a chat function, pass it to `gr.ChatInterface`, launch. Gradio handles everything else.

In [ ]:
# Minimal Gradio chat app
# -----------------------
MINIMAL_APP = '''
import gradio as gr
from agents import Agent, Runner
from config import MODEL

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list) -> str:
    """Handle one turn of the conversation."""

    result = await Runner.run(agent, input=message)

    return result.final_output


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent",
    description="Chat with your AI agent."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Minimal Gradio app:")
print(MINIMAL_APP)

### 💡 Key Takeaway

`gr.ChatInterface` expects a function that takes `message` and `history` and returns a string.

For this simple stateless agent, the agent definition stays the same. You're just changing what calls it.

⚠️ **History vs memory:** `gr.ChatInterface` passes `history` to your chat function, but this is UI state: it shows past messages in the interface.

The agent itself is stateless unless you reconstruct history into the input or use sessions (Lesson 17).

Use this pattern for lightweight short-term context.

Use sessions when you want the SDK to manage conversation state.

Gradio passes `history` as a list of `{'role': 'user'|'assistant', 'content': text}` dicts (most recent last).

To include recent context, build the input from history before passing it to the agent:

    context = "\n".join([f"User: {h['content']}" if h["role"] == "user" else f"Assistant: {h['content']}" for h in history[-6:]])
    full_input = f"{context}\nUser: {message}" if context else message
    result = await Runner.run(agent, input=full_input)

For persistent memory across sessions, use Lesson 18 instead.

---

## 📡 Part 3: Streaming Responses

The minimal app waits for the full response. Streaming shows text as it's generated: progress appears sooner, though generation takes the same time.

Gradio supports streaming via Python generators: use `Runner.run_streamed()` and `yield` each text chunk to Gradio instead of returning the full response.

In [ ]:
# Streaming Gradio app
# --------------------
STREAMING_APP = '''
import gradio as gr
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list):
    """Stream the agent response chunk by chunk."""
    response_text = ""

    result = Runner.run_streamed(agent, input=message)

    async for event in result.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                response_text += data.delta
                yield response_text  # Gradio updates the UI on each yield


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent (Streaming)",
    description="Responses appear as they arrive."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Streaming Gradio app:")
print(STREAMING_APP)

### 💡 Key Takeaway

Gradio treats any generator function as a streaming source: every `yield` updates the chat window.

`Runner.run_streamed()` produces text deltas as the model generates them.

---

## 🔧 Part 4: Tool Visibility

When an agent calls a tool, the user sees nothing. Adding a brief status update makes the experience feel responsive.

This matters most when tools are slow. Without a status update, users often assume the app froze or the request failed.

The pattern: yield a status line before the tool result arrives, then replace it with the final response.

In [ ]:
# Streaming with tool visibility
# --------------------------------
TOOL_VISIBILITY_APP = '''
import gradio as gr
from agents import Agent, Runner, function_tool, ToolCallItem
from agents.stream_events import RawResponsesStreamEvent, RunItemStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL


@function_tool
def get_weather(location: str) -> str:
    """Demo weather tool: returns synthetic data, not a real forecast."""
    return f"Weather in {location} (demo data): 72°F and sunny"


agent = Agent(
    name="WeatherAssistant",
    instructions="Help users with weather questions.",
    model=MODEL,
    tools=[get_weather]
)


async def chat(message: str, history: list):
    """Stream responses and show tool call status."""
    response_text = ""

    result = Runner.run_streamed(agent, input=message)

    async for event in result.stream_events():
        if isinstance(event, RunItemStreamEvent):
            # Tool call starting: show status to user
            item = event.item
            if isinstance(item, ToolCallItem):
                yield f"🔧 Calling {item.raw_item.name}..."

        elif isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                response_text += data.delta
                yield response_text


demo = gr.ChatInterface(
    fn=chat,
    title="Weather Agent (Demo)",
    description="Synthetic demo weather: not real forecasts."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Tool visibility app:")
print(TOOL_VISIBILITY_APP)

### 💡 Key Takeaway

`RunItemStreamEvent` fires for various agent activities, including when a tool call starts.

Yielding a status string at that point surfaces tool progress in the chat window, which is then replaced by the actual response as text deltas arrive.

`item.raw_item.name` fits function tools like this one: other tool-call types carry differently shaped raw items.

---

## ⚠️ Part 5: Handling Failure States

Agent calls can fail: tool timeouts, API errors, or model issues.

A Gradio app that crashes with a stack trace feels broken. A Gradio app that returns a friendly message feels like a product.

Wrap the agent call in a try/except: log the details server-side, and yield a generic message to the user.

In [ ]:
# Streaming app with failure handling
# -------------------------------------
FAILURE_HANDLING_APP = '''
import logging

import gradio as gr
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL

logger = logging.getLogger(__name__)

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list):
    """Stream responses with graceful error handling."""
    try:
        response_text = ""

        result = Runner.run_streamed(agent, input=message)

        async for event in result.stream_events():
            if isinstance(event, RawResponsesStreamEvent):
                data = event.data
                if isinstance(data, ResponseTextDeltaEvent):
                    response_text += data.delta
                    yield response_text

    except Exception:
        # Log the details server-side, show the user a generic message
        logger.exception("Agent request failed")
        yield "⚠️ Something went wrong. Please try again."


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent",
    description="Chat with your AI agent."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Failure handling app:")
print(FAILURE_HANDLING_APP)

### 💡 Key Takeaway

Gradio's generator pattern means any `yield` (including an error message) updates the chat window.

Catching exceptions at the chat function level keeps errors contained: the user sees a generic message, the details go to the server log, and later requests recover if the underlying problem was transient (a missing API key will keep failing until fixed).

---

## 🚀 Part 6: Deploying to Hugging Face Spaces

Hugging Face Spaces is a hosted environment for Gradio apps with a free tier.

To deploy:

1. Sign in at [huggingface.co](https://huggingface.co)
2. Click **New Space**, give it a name, choose **Gradio** as the SDK, and pick a visibility
3. Upload your files via the web UI drag-and-drop, or `git push` to the Space's repo
4. Add your `OPENAI_API_KEY` as a secret under **Settings → Variables and secrets** (set `MODEL` as a variable there if you want to override it)
5. The Space builds automatically and gets a shareable URL

**The minimum file set:**

```
my_agent_space/
├── README.md           # Space config: sdk, sdk_version, app_file
├── app.py              # your Gradio app (the chat function + demo.launch())
├── requirements.txt    # pinned deps (gradio comes from the Space runtime)
└── .gitignore          # include .env
```

The Space's `README.md` YAML header configures the runtime: keep it, and pin `sdk_version` to the Gradio version you tested.

**No `.env` file.** Spaces stores secrets in **Settings → Variables and secrets**.

Your code reads them the same way: `os.getenv("OPENAI_API_KEY")`.

⚠️ **Your key pays for every chat.** A public Space runs on your `OPENAI_API_KEY`: anyone with the URL consumes your quota. Keep the Space private, add access control (`demo.launch(auth=...)` with credentials from secrets, or Space visibility settings), and watch your usage dashboard before sharing broadly.

In [ ]:
# Deployment file templates
# --------------------------

APP_PY = '''
# app.py: complete Gradio app for Hugging Face Spaces
import logging
import os

import gradio as gr
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

logger = logging.getLogger(__name__)

MODEL = os.getenv("MODEL", "gpt-5-mini")

# Fail fast with a clear message instead of failing on the first chat
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not set. Add it under Settings -> Variables and secrets")

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list):
    # Single-turn: each message is independent (see Part 2 for history and sessions)
    try:
        response_text = ""

        result = Runner.run_streamed(agent, input=message)

        async for event in result.stream_events():
            if isinstance(event, RawResponsesStreamEvent):
                data = event.data
                if isinstance(data, ResponseTextDeltaEvent):
                    response_text += data.delta
                    yield response_text
    except Exception:
        logger.exception("Agent request failed")
        yield "⚠️ Something went wrong. Please try again."


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent",
    description="Single-turn demo: each message is independent."
)

demo.launch()
'''

README_MD = '''
---
title: My Agent
sdk: gradio
sdk_version: 6.20.0
app_file: app.py
---

Chat with my agent.
'''

REQUIREMENTS_TXT = '''
openai-agents==0.13.0
openai==2.29.0
'''

print("app.py:")
print(APP_PY)
print("README.md:")
print(README_MD)
print("requirements.txt:")
print(REQUIREMENTS_TXT)

### ✅ Deployment Checklist

Note: `APP_PY` reads `MODEL` from `os.getenv()` so you can set it as a Space variable.

You can still use the Lesson 28 module structure on Spaces.

This template combines streaming and failure handling.

If your agent uses tools, copy the `RunItemStreamEvent` block from Part 4 into this `chat()` function so tool activity is visible in the UI.

Before pushing to Hugging Face Spaces:

- `app.py` calls `demo.launch()` without host or port arguments (Spaces handles those). Add `auth=` only when using Gradio authentication, with credentials from secrets, not source code

- `README.md` keeps the YAML header, with `sdk_version` pinned to the Gradio version you tested

- `requirements.txt` pins `openai-agents==0.13.0` and `openai==2.29.0`. Spaces installs them on startup (Gradio comes from the runtime)

- `OPENAI_API_KEY` is a secret under Settings → Variables and secrets, not in a committed file

- `.env` is in `.gitignore`. Never commit API keys

- Decide who can use it: private Space, `launch(auth=...)`, or public with your quota on the line

- Test locally before pushing: set `OPENAI_API_KEY` in your shell (`export OPENAI_API_KEY=...`), then `python app.py`

---

## 💪 Practice Exercises

### Exercise 1: Streaming Chat App

*Covers: Gradio, building a streaming chat interface*

Build a streaming Gradio app around one of the agents you built in this course.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Streaming Chat App
# --------------------------------------------------------------
# Objective: Wrap an existing agent in a streaming Gradio interface.

# TODO 1: Choose an agent without approval flows or MCP servers
#          (a tools agent from Weeks 1-2, or one you built in an exercise).
#          The Lesson 24 and 27 agents need approval handling and server
#          lifecycle adaptations that don't fit this template: input()-based
#          approval does not work inside a web app.

# TODO 2: Create a streaming chat function using Runner.run_streamed()
#          that yields response_text on each text delta

# TODO 3: Wrap it in gr.ChatInterface with a title and description

# TODO 4: Add a try/except block: log the details server-side
#          (logger.exception) and yield a generic friendly message

# TODO 5: Test it by running demo.launch() and chatting with the agent

# --- Write your code below this line ---

### Exercise 2: Deploy to Hugging Face Spaces

*Covers: end-to-end deployment to Hugging Face Spaces*

Take the app from Exercise 1 and prepare it for deployment.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Prepare for Deployment
# --------------------------------------------------------------
# Objective: Create the files needed to deploy your agent to HF Spaces.

# TODO 1: Create app.py with your streaming chat function
#            no .env loading, reads OPENAI_API_KEY from os.getenv()
#            and raises at startup if it is missing
#            calls demo.launch() without host or port arguments

# TODO 2: Create requirements.txt pinning openai-agents==0.13.0
#            and openai==2.29.0
#            (Gradio comes from the Space runtime via README.md)

# TODO 3: Keep the Space's README.md YAML header: pin sdk_version to
#            your tested Gradio version, and create .gitignore with .env

# TODO 4: (Optional) Create a Hugging Face Space, choose its visibility,
#            add OPENAI_API_KEY under Settings → Variables and secrets,
#            and push your files. Remember: a public Space spends YOUR
#            API quota.
#            Your agent will be live at: https://huggingface.co/spaces/<your-username>/<space-name>

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Gradio turns an agent into a chat app with one function:**

- `gr.ChatInterface` handles the UI and visible chat history

- The agent is still stateless unless you pass history into the input or use sessions (Lesson 17)

- Your chat function just takes `message` and `history`: return a string or yield partial strings
<br>
<br>

**Streaming requires one change and one bridge:**

- Change `Runner.run()` to `Runner.run_streamed()`

- Bridge: accumulate text deltas, `yield` the growing string on each delta

- Gradio updates the UI on every `yield` automatically
<br>
<br>

**Tool visibility and failure handling keep the app feeling solid:**

- Yield a status message when a tool call starts: users see something instead of silence

- Tool failures and model errors are caught at the app level: yield a friendly message, never a stack trace
<br>
<br>

**Deployment to Hugging Face Spaces needs a small file set:**

- `app.py` (calls `demo.launch()` without host or port arguments), `requirements.txt` (pinned), and the Space `README.md` (sdk_version, app_file)

- `OPENAI_API_KEY` as a secret under Settings → Variables and secrets. Never commit API keys

- Choose visibility and access control deliberately: a public Space runs on your quota

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-30-deploying-with-gradio)

---

## 🎉 Course Complete

You've gone from a 5-line agent to a deployed chat interface, with tools, memory, guardrails, multi-agent orchestration, human oversight, MCP integration, and a live shareable URL.

**What you've built:**

- Four complete capstone projects: Research Agent, Multi-Agent Research Team, Production-Pattern Customer Service Agent, MCP-Powered Personal Assistant

- A production-readiness toolkit: tools, structured outputs, sessions, guardrails, tracing, MCP integration, and deployment

- The judgment to choose the right level of complexity, and the habit of justifying it with data

The OpenAI Agents SDK is moving fast.

Keep an eye on the official documentation at `openai.github.io/openai-agents-python`. New capabilities appear regularly.

**Build something. Evaluate it. Improve it. That's the loop.**

---